# Component injection: the components of a design, and random flip-flop upsets in one component

This example shows the components of the design hierarchy as a tree, with the number of flip-flops in each component. Then it injects upsets only into the flip-flops of one of these components, and the LEDs show the effect. It uses the design of [example 2](../example2/README.md) (`bscan_adder_top`)

A component is one or more whole levels of the net paths in the LL file. For example, the net `bscan_u1_operands/bscan_interface_read_write_uX/data_user_reg_data_in[2]` is inside the component `bscan_u1_operands`. The last level, `data_user_reg_data_in[2]`, is the net. It is not a component. A part of a name is not a component: `bscan_u1` does not select this net.

### Example of the hierarchy

Using `bscan_adder_top` as an example.

```
3 top-level components:

bscan_u1_operands (20 flip-flops)
`-- bscan_interface_read_write_uX (20 flip-flops)
bscan_u2_simplex_adder_res (5 flip-flops)
`-- bscan_interface_read_write_uX (5 flip-flops)
bscan_u3_tmr_adder_res (5 flip-flops)
`-- bscan_interface_read_write_uX (5 flip-flops)
```

## Sequence

1. The script uploads the bitstream and the LL file, and programs the FPGA. The server reads the LL file when it programs the FPGA. Thus, `dev.components()` needs a programmed design.
2. **Part 1.** The script gets the components with `dev.components()`. The result is a list of the top-level components. Each component holds its sub components.
3. The script prints each top-level component with `print(component)`. This shows the component and all its sub components as a tree. Then the script prints the full path of each component, with its number of flip-flops.
4. **Part 2.** The script finds `COMPONENT` (`bscan_u1_operands`) in the components. If the design has no such component, the script stops. If it has, the script prints the number of flip-flops in the component.
5. The script injects one upset each 0.5 s with `dev.inject_random(TargetMemory.FF, component=COMPONENT)`, until you press Enter. The server selects a random flip-flop of the component, also in its sub components.
6. The script programs the FPGA again, to clear the upsets.

For each upset, the script prints the location of the flip-flop (FAR, word and bit) and the net that the flip-flop drives. The net is always inside `COMPONENT`.

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device, in `with` statements that release the device and close the connection. |
| `Design(bit, ll)`, `dev.uploadDesignToServer(design)`, `dev.programFPGA(design)` | Give the design files, upload them to the server and program the FPGA. |
| `dev.configure_seed(0)` | Set the seed of the random injections. The same seed gives the same upsets in each run. |
| `dev.components()` | Part 1: get the components of the design as a tree, with the number of flip-flops in each component. |
| `dev.inject_random(TargetMemory.FF, component=COMPONENT)` | Part 2: inject one upset into a random flip-flop of one component. The report gives the net of the flip-flop (`net_name`). |

| Constant | Default | Description |
| --- | --- | --- |
| `COMPONENT` | `"bscan_u1_operands"` | The full path of the component, as part 1 prints it. |
| `INJECTION_PERIOD_S` | `0.5` | The time between two upsets, in seconds. |

## Result

The script prints the tree, the full paths, and then one line for each upset: the number of the upset, the FAR, the word, the bit and the net. In `targeted_top`, the top-level components are `clk_gen_inst` (the clock generator), `fast_heartbeat` and `slow_heartbeat`.

The number of flip-flops and the locations depend on the build of your design. The numbers come from the LL file. If you exclude pblocks or frames, `dev.inject_random()` can select fewer flip-flops of a component than the tree shows. The script sets the seed (`dev.configure_seed(0)`). Thus, with the same design, each run injects into the same sequence of flip-flops.

If no flip-flop of the LL file is inside a component, the script prints a message and stops.

The script does not correct the upsets, and it does not stop the clock of the design during the injection. If your design must not run during the injection, stop its clock first, as [example 3](../example3/README.md) does. At the end, the script programs the FPGA again to clear the upsets.

If you stop the script with Ctrl+C, the configuration of the device can stay incorrect. In that case, program the FPGA again before you do other work.
## Run the example

### 0: Install the module

This will restart the kernel, a popup will show up, just ignore it.

In [1]:
%pip install ../../api/python

import IPython; IPython.get_ipython().kernel.do_shutdown(restart=True)

Defaulting to user installation because normal site-packages is not writeable
Processing /home/wallmenis/git/sk-raft-client/api/python
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for skraft: filename=skraft-1.0.0-py3-none-any.whl size=45974 sha256=3b94e7b15d9638ae92483345a15864a616483b0d5073bfd83a625d796367efca
  Stored in directory: /tmp/pip-ephem-wheel-cache-ehtdh3pc/wheels/cc/7e/0f/603cd9eaf8e594f1c766c921daeeef2440d17e0cbb287bfb57
Successfully built skraft
  Attempting uninstall: skraft
    Found existing installation: skraft 1.0.0
    Uninstalling skraft-1.0.0:
      Successfully uninstalled skraft-1.0.0
Note: you may need to restart the kernel to use updated packages.


{'status': 'ok', 'restart': True}

### 1: Extract the design and bitstream files

On Linux

In [1]:
%%bash
cd ../hw/design_examples/
./unzip_designs.sh

Design files extracted to /home/wallmenis/git/sk-raft-client/get_started/hw/design_examples


On Windows

In [ ]:
%%script pwsh
cd ..\hw\design_examples\
.\unzip_designs.bat

### 2: Add the imports

In [2]:
import random
import time

from skraft import Design, TargetMemory, ReadbackOption, SkraftError, SkraftFpga, BscanPrimitive

SK_RAFT_SERVER_IP = "127.0.0.1"
SK_RAFT_SERVER_PORT = 50051

### 3: Start the server

In [7]:
%%script bash --bg
sk-raft

### 4: List devices
Connect the devices you want to test before running list devices.

In [3]:
def main() -> int:
    try:
        # SK-RAFT: open the connection. Then list the connected boards.
        with SkraftFpga.connect(ip=SK_RAFT_SERVER_IP,
                                port=SK_RAFT_SERVER_PORT) as client:
            devices = client.get_devices()
    except SkraftError as e:
        print(f"Cannot list the boards: {e}")
        return 1
    except KeyboardInterrupt:
        print("Interrupted.")
        return 130

    if not devices:
        print("No board is connected to the SK-RAFT server.")
        return 1
    print(f"{'SERIAL NUMBER':16s} PART NUMBER")
    for device in devices:
        print(f"{device.serial_number:16s} {device.part_number}")
    return 0


if __name__ == "__main__":
    main()

SERIAL NUMBER    PART NUMBER
210183BA2524A    XC7A35T


### 5: Setup the code
Don't forget to change the Serial Number (`SERIAL_NUMBER`) and Board (`BOARD`) to the ones you desire from the list!

In [9]:
SERIAL_NUMBER = "210183BA2524A"

# The design for both parts: a flip-flop shift register and a LUTRAM.
# The script gives all three files at the start:
#   The bitstream (.bit): to program the device.
#   The XDC file (.xdc): to find the pblocks.
#   The LL file (.ll): to find the BRAM/FF/LUTRAM bits that the design uses.
# uploadDesignToServer() sends the files to the server. The client sends
# only the file name of each path, not the directory.
#
# BOARD is the board of your target device: "basys3" or "zedboard", or
# the name of your own board (see hw/README.md). The script reads the
# design files from hw/design_examples/<BOARD>/ff_lut_top/. Extract
# design_examples.zip there, or build the design with create_project.tcl.
# The paths are relative to src/. Run the script from that folder.
BOARD = "basys3"
DESIGN_DIR = f"../hw/design_examples/{BOARD}/bscan_adder_top"

design = Design(
    bit=f"{DESIGN_DIR}/bscan_adder_top.bit",
    ll=f"{DESIGN_DIR}/bscan_adder_top.ll",
)


# The component to test: the full path of a component, as part 1 shows
# it. This one is the slow heartbeat (its counter and its output).
COMPONENT = "bscan_u1_operands"

# The time between two upsets, in seconds.
INJECTION_PERIOD_S = 0.5

### 6: Define some helper functions

In [10]:
# Go through all the components of the tree, each component before its
# sub components.
def all_components(
        components: list[FpgaComponent]) -> Iterator[FpgaComponent]:
    for component in components:
        yield component
        yield from all_components(component.children)


# Find a component by its full path in the tree of the components.
def find_component(components: list[FpgaComponent],
                   path: str) -> Optional[FpgaComponent]:
    for component in all_components(components):
        if component.path == path:
            return component
    return None

### 7: Connect to the server...

...and select the board

In [11]:
client = SkraftFpga.connect(ip=SK_RAFT_SERVER_IP, port=SK_RAFT_SERVER_PORT)
dev = client.attach(SERIAL_NUMBER)

### 8: Program the board...
...and upload the design to the server because it needs it for some operations.

In [12]:
dev.uploadDesignToServer(design)
dev.programFPGA(design)
print(f"programmed {SERIAL_NUMBER} with {design.bit}")

programmed 210183BA2524A with ../hw/design_examples/basys3/bscan_adder_top/bscan_adder_top.bit


### 9: Run the experiment!
The `N_OF_INJECTIONS` is the number of fault injections the experiment will do.

In [14]:
N_OF_INJECTIONS = 100


# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)

# Part 1. SK-RAFT: get the tree of the components.
components = dev.components()
if not components:
    print("No flip-flop of the LL file is inside a component. The design has its flip-flops at the top level only.")
    raise SystemExit("No FF")

# print() shows a component and all its sub components.
print(f"\n{len(components)} top-level components:\n")
for component in components:
    print(component)

# The full paths, to copy into COMPONENT.
print("\nComponent paths for inject_random(TargetMemory.FF, component=<path>):\n")
width = max(len(component.path) for component in all_components(components))
for component in all_components(components):
    count = component.flip_flop_count
    unit = "flip-flop" if count == 1 else "flip-flops"
    print(f"  {component.path:<{width}}  {count:>6} {unit}")

# Part 2. Make sure that the component exists and holds
# flip-flops.
component = find_component(components, COMPONENT)
if component is None:
    print(f"\nNo component {COMPONENT!r} in the LL file. Set COMPONENT to one of the paths above.")
    raise SystemExit("no component")
print(f"\n{component.path}: {component.flip_flop_count} flip-flops.")

print("Injecting random upsets, not corrected. Press Enter to stop.")
print("     #  FAR       WORD  BIT  NET")
iterator = N_OF_INJECTIONS
n = 0
while iterator != 0:
    # Injection guideline (application memory): each upset
    # is a readback-modify-writeback of full frames. If your
    # design runs, stop its clock during the injection, as
    # example 3 does.
    # SK-RAFT: flip one flip-flop bit of the component.
    report = dev.inject_random(TargetMemory.FF, component=COMPONENT)
    # net_name is the net that the flip-flop drives, from
    # the LL file. It is always inside COMPONENT.
    print(f"{n + 1:>6}  {report.far:08x}  {report.word_index:>4}  {report.bit_index:>3}  {report.net_name}")
    n += 1
    iterator -= 1
    # Wait to control the number of injections.
    time.sleep(INJECTION_PERIOD_S)

print(f"{n} upsets injected into {COMPONENT}.")


3 top-level components:

bscan_u1_operands (20 flip-flops)
`-- bscan_interface_read_write_uX (20 flip-flops)
bscan_u2_simplex_adder_res (5 flip-flops)
`-- bscan_interface_read_write_uX (5 flip-flops)
bscan_u3_tmr_adder_res (5 flip-flops)
`-- bscan_interface_read_write_uX (5 flip-flops)

Component paths for inject_random(TargetMemory.FF, component=<path>):

  bscan_u1_operands                                             20 flip-flops
  bscan_u1_operands/bscan_interface_read_write_uX               20 flip-flops
  bscan_u2_simplex_adder_res                                     5 flip-flops
  bscan_u2_simplex_adder_res/bscan_interface_read_write_uX       5 flip-flops
  bscan_u3_tmr_adder_res                                         5 flip-flops
  bscan_u3_tmr_adder_res/bscan_interface_read_write_uX           5 flip-flops

bscan_u1_operands: 20 flip-flops.
Injecting random upsets, not corrected. Press Enter to stop.
     #  FAR       WORD  BIT  NET
     1  00400a1f    62    1  bscan_u1_opera

In [15]:
print("Re-programming to clear the device.")
dev.programFPGA(design)

Re-programming to clear the device.
